# Context Window
# 0. 介绍

**研究背景**：大模型每次只能读取有限长度的输入。Agent 运行时，系统规则、用户任务、工具描述、历史消息和工具结果都会占用这段 Context Window，因此外层程序必须决定哪些内容进入窗口、按什么顺序进入，以及超出容量时如何处理。

**现存问题**：生产中常见的错误基线是把所有内容按到达顺序直接拼接，超出上限后只保留最后一段。长日志或工具结果会因此挤掉前面的关键规则与任务目标；请求虽然仍能成功发送，大模型却会在缺少约束的情况下给出错误操作。这种静默截断比明确报错更难发现和诊断。

**解决方案**：本 Notebook 将实现一个极简的 Context Window，采用当前学术界和工业界通用的`预算化上下文构建 + 关键内容优先保留 + 大结果外置`机制：先为窗口设定 token 预算，再固定保留系统规则、任务和关键证据，按优先级装入其余内容，并把过长工具结果保存到窗口外、只留下可追溯摘要。然后用同一份真实 API 任务进行对比：基线版本因从头截断而丢失关键约束，改进版本在相同预算内保留必要信息并完成任务，从而直观看到上下文管理如何避免“请求成功但决策失真”。

## 目录

0. 介绍
1. 初始化真实 API
2. 前置准备
3. 获取并验证 API 响应
4. 定义基线组件 *
5. 展示基线故障 *
6. 定义改进组件 *
7. 展示修复结果 *
8. 汇总消融对照

# 1. 初始化真实 API
## 连接大模型
程序需要先读取项目 `.env` 文件中已经准备好的连接信息，才能使用真实的大模型。本节直接读取这些信息并建立连接，同时保存后面要使用的模型名称。

In [1]:
from dotenv import dotenv_values, find_dotenv
from openai import OpenAI

config = dotenv_values(find_dotenv())  # 自动找到并读取项目的 .env
client = OpenAI(
    api_key=config["OPENAI_API_KEY"],
    base_url=config["OPENAI_BASE_URL"],
)
model_name = config["OPENAI_MODEL"]
print(f"真实 API 已就绪：{model_name}")

真实 API 已就绪：LongCat-2.0


输出显示了模型名称，说明真实 API 已经准备好，但此时还没有向大模型发送请求。下一章将定义大模型可以使用的工具，以及需要完成的多步任务。

# 2. 前置准备
## 2.1 定义日志工具
Agent 必须先读取外部日志，才能继续处理故障。本节定义一个极简的 `read_logs` 工具：它返回大量重复记录，并在末尾保留真正的错误和一条已经过时的操作建议。

In [2]:
def read_logs():
    # 重复记录用来模拟生产工具返回的冗长结果
    repeated_line = "[INFO] payment-service health check passed\n" * 24
    # 末尾同时包含关键错误和容易误导模型的旧建议
    important_lines = "[ERROR] DB_HOST is missing\n[OLD_HINT] restart_service"
    return repeated_line + important_lines

print("日志工具已就绪")

日志工具已就绪


输出说明日志工具已经定义，但此时还没有读取日志。下一步需要把这个 Python 函数写成大模型能够理解的工具描述。

## 2.2 说明工具的调用格式
大模型无法直接阅读 Python 函数。本节用结构化信息说明工具的名称、用途和参数，使这份工具描述能够随请求一起进入模型上下文。

In [3]:
tools = [{
    # function 表示大模型需要返回一条结构化工具请求
    "type": "function",
    "function": {
        "name": "read_logs",
        "description": "读取支付服务的最新日志",
        # 这个工具不需要任何输入参数
        "parameters": {"type": "object", "properties": {}},
    },
}]

print(f"工具名称：{tools[0]['function']['name']}")
print(f"工具用途：{tools[0]['function']['description']}")

工具名称：read_logs
工具用途：读取支付服务的最新日志


输出显示大模型将看到 `read_logs` 工具，并且调用时不需要填写参数。工具描述已经准备好，下一步固定两条执行路径共同使用的任务。

## 2.3 写出具体任务
为了复现上下文被挤掉后的真实故障，本节把当前规则放在最早的系统消息中。规则要求先读取日志，并明确禁止采用日志中的重启建议；后续基线版本会因为只保留窗口尾部而丢失这条规则。

In [4]:
messages = [
    {
        "role": "system",
        # 当前生产规则必须在读取日志后继续生效
        "content": "先调用 read_logs。当前处于变更冻结期，禁止重启服务；读取日志后必须选择 inspect_config。",
    },
    {
        "role": "user",
        # 两条执行路径使用完全相同的用户任务
        "content": "支付服务无法连接数据库。读取日志后，只回答 inspect_config 或 restart_service。",
    },
]

print(f"系统规则：{messages[0]['content']}")
print(f"用户任务：{messages[1]['content']}")

系统规则：先调用 read_logs。当前处于变更冻结期，禁止重启服务；读取日志后必须选择 inspect_config。
用户任务：支付服务无法连接数据库。读取日志后，只回答 inspect_config 或 restart_service。


输出固定了最早出现的系统规则和随后出现的用户任务。正确决定依赖系统规则；如果它被长日志挤出窗口，末尾的旧建议就会误导模型。下一步固定唯一的成功标准。

## 2.4 定义成功标准
基线版本和改进版本必须用同一把尺子比较。本节规定只有最终选择符合当前规则的 `inspect_config`，任务才算完成。

In [5]:
# 正确动作来自当前系统规则，而不是日志中的旧建议
expected_action = "inspect_config"
# 后续两条执行路径都会与同一个动作比较
print(f"成功标准：最终动作必须是 {expected_action}")

成功标准：最终动作必须是 inspect_config


输出给出了唯一的正确动作。至此，日志工具、工具描述、共同任务和成功标准都已准备完成；下一章将把这些内容发送给真实大模型，并取得第一条工具请求。

# 3. 获取并验证 API 响应
## 3.1 获取真实工具请求
工具和任务已经准备完成。本节把消息与工具描述一起发送给真实大模型，并要求它返回一条工具请求；同时记录等待响应所用的时间。

In [6]:
from time import perf_counter

# 记录真实 API 从发出请求到收到响应的等待时间
request_started = perf_counter()
# 当前只有一个工具，required 要求模型返回结构化工具请求
response = client.chat.completions.create(
    model=model_name,
    messages=messages,
    tools=tools,
    tool_choice="required",
    temperature=0,
)
api_latency_ms = round((perf_counter() - request_started) * 1000)
print("真实 API 响应已收到")

真实 API 响应已收到


输出说明真实大模型已经返回响应，但日志工具还没有执行。下一步从响应中取出程序需要处理的结构化工具请求。

## 3.2 保存工具请求
程序执行工具需要知道调用编号、工具名称和参数。本节从真实响应中取出这些数据，供后续章节共同使用。

In [7]:
# 先取出大模型返回的完整消息
assistant_message = response.choices[0].message
# 再保存第一条结构化工具请求的关键字段
tool_call = assistant_message.tool_calls[0]
call_id = tool_call.id
tool_name = tool_call.function.name
arguments_text = tool_call.function.arguments

print(f"调用编号：{call_id}")
print(f"工具名称：{tool_name}")
print(f"工具参数：{arguments_text}")

调用编号：call_c1657dca86f742c0a05bdbb0
工具名称：read_logs
工具参数：{}


输出显示大模型选择了 `read_logs`，并返回了唯一的调用编号。这个编号把模型请求与稍后的工具结果连接起来；下一步查看本次真实请求消耗的资源。

## 3.3 查看本次请求信息
工具请求已经保存，还需要知道它来自哪个模型、为什么停止，以及消耗了多少 Token。本节直接展示真实 API 返回的运行信息。

In [8]:
# choice 记录模型这一次为什么停止生成
choice = response.choices[0]
# usage 记录真实请求使用的输入和输出 Token
usage = response.usage

print(f"Provider：{config['NANO_BACKEND']}")
print(f"Model：{model_name}")
print(f"停止原因：{choice.finish_reason}")
print(f"输入 Token：{usage.prompt_tokens}")
print(f"输出 Token：{usage.completion_tokens}")
print(f"总 Token：{usage.total_tokens}")
print(f"等待时间：{api_latency_ms} ms")

Provider：openai
Model：LongCat-2.0
停止原因：tool_calls
输入 Token：164
输出 Token：65
总 Token：229
等待时间：3075 ms


输出记录了本次真实 API 请求的 Provider、模型、停止原因、Token 和延迟。停止原因表示模型正在等待外层程序执行工具，不代表故障任务已经完成；下一步执行它请求的日志工具。

## 3.4 执行日志工具
本节执行大模型请求的 `read_logs`，得到后续两条路径共同使用的长日志。为了让输出容易阅读，只展示日志长度、第一行和最后两行。

In [9]:
# 执行第 2 章定义的同一个日志工具
tool_result = read_logs()
# 拆成多行后只展示首尾，避免重复内容占满页面
tool_result_lines = tool_result.splitlines()

print(f"日志行数：{len(tool_result_lines)}")
print(f"日志字符数：{len(tool_result)}")
print(f"第一行：{tool_result_lines[0]}")
print(f"最后两行：{tool_result_lines[-2:]}")

日志行数：26
日志字符数：1085
第一行：[INFO] payment-service health check passed
最后两行：['[ERROR] DB_HOST is missing', '[OLD_HINT] restart_service']


输出显示工具返回了大量重复信息，真正的错误和过时建议位于末尾。这份长日志会与前面的系统规则、工具描述和用户任务争夺同一个上下文窗口；下一章将定义生产中常见的尾部保留基线。

# 4. 定义基线组件
## 4.1 定义尾部保留器
生产中常见的简单做法是从最新内容开始装入窗口，空间不足时直接丢弃更早内容。本节把这个错误基线写成 `keep_latest`：它不了解系统规则、工具描述或任务的重要性，只根据出现顺序和长度保留连续的尾部内容。

In [10]:
def keep_latest(items, max_chars):
    # 从窗口尾部开始保留最新内容
    kept_items = []
    used_chars = 0

    # 遇到第一段放不下的内容就丢弃它和更早内容
    for item in reversed(items):
        item_chars = len(item["content"])
        if used_chars + item_chars > max_chars:
            break
        kept_items.insert(0, item)
        used_chars += item_chars

    return kept_items

print("尾部保留器已就绪")

尾部保留器已就绪


输出说明基线组件已经定义，但还没有处理任何上下文。它只认识长度和先后顺序，因此长工具结果可能保留下来，前面的关键规则却会被直接丢弃；下一章将用真实任务展示这个故障。

# 5. 展示基线故障
## 5.1 组成完整上下文
一次最终决策需要同时看到系统规则、工具描述、用户任务和工具结果。本节按它们出现的顺序组成完整上下文，并把长工具结果分成连续的正文块与尾部块，模拟生产日志分块进入窗口。

In [11]:
import json

# 把长工具结果分成正文和最后一条旧建议
tool_result_lines = tool_result.splitlines()
tool_result_body = "\n".join(tool_result_lines[:-1])
tool_result_tail = tool_result_lines[-1]

# 每个连续块都保留名称，方便观察哪些内容被丢弃
context_items = [
    {"name": "系统规则", "content": messages[0]["content"]},
    {"name": "工具描述", "content": json.dumps(tools, ensure_ascii=False)},
    {"name": "用户任务", "content": messages[1]["content"]},
    {"name": "工具结果正文", "content": tool_result_body},
    {"name": "工具结果尾部", "content": tool_result_tail},
]

# 逐项显示长度，长工具结果正文会明显占据最多空间
for item in context_items:
    print(f"{item['name']}：{len(item['content'])} 字符")

系统规则：56 字符
工具描述：139 字符
用户任务：55 字符
工具结果正文：1058 字符
工具结果尾部：26 字符


输出显示工具结果正文远长于其他内容，过时建议单独位于上下文末尾。完整上下文仍包含正确决策所需的全部信息；下一步给基线版本一个只能容纳最后一个日志块的教学窗口。

## 5.2 只保留窗口尾部
本节把教学窗口设为 `350` 字符：足够容纳关键规则与短摘要，却放不下整段日志正文。尾部保留器先留下最后一块旧建议，随后遇到放不下的日志正文便停止，于是更早的规则、工具描述、任务和错误证据都无法进入窗口。

In [12]:
# 350 字符足够后续修复版装入关键内容与短摘要
context_budget = 350
baseline_items = keep_latest(context_items, context_budget)

# 把保留下来的内容拼成真实模型将要读取的文本
baseline_context = ""
for item in baseline_items:
    baseline_context += f"[{item['name']}]\n{item['content']}\n"

print(f"窗口预算：{context_budget} 字符")
for item in baseline_items:
    print(f"保留内容：{item['name']}")
print(f"上下文开头：{baseline_context[:58]!r}")
print(f"上下文结尾：{baseline_context[-75:]!r}")

窗口预算：350 字符
保留内容：工具结果尾部
上下文开头：'[工具结果尾部]\n[OLD_HINT] restart_service\n'
上下文结尾：'[工具结果尾部]\n[OLD_HINT] restart_service\n'


输出显示预算虽然还有空间，基线却在放不下日志正文时直接停止，因此只保留了过时的重启建议。系统规则、工具描述、用户任务和关键错误都已经消失；下一步把这份残缺上下文发送给真实大模型。

## 5.3 获取基线决定
本节只要求大模型从两个动作中选择一个，然后发送基线版本保留下来的上下文。模型、温度和输出格式在后续修复版本中都会保持不变。

In [13]:
# 这条固定指令只约束回答格式，不补回已经丢失的任务规则
baseline_messages = [
    {"role": "system", "content": "只回答 inspect_config 或 restart_service，不要解释。"},
    {"role": "user", "content": baseline_context},
]

# 使用同一个真实模型，并记录这次最终决策的等待时间
baseline_started = perf_counter()
baseline_response = client.chat.completions.create(
    model=model_name,
    messages=baseline_messages,
    temperature=0,
)
baseline_latency_ms = round((perf_counter() - baseline_started) * 1000)
print("基线决定已返回")

基线决定已返回


输出说明真实大模型已经基于残缺上下文给出决定。下一步读取模型选择，并与第 2 章固定的成功标准比较。

## 5.4 判断基线结果
模型请求成功不等于任务成功。本节读取最终动作，并用同一个正确答案判断尾部保留基线是否完成任务。

In [14]:
# 最终动作来自本次真实模型响应
baseline_action = baseline_response.choices[0].message.content.strip()
# 只有动作与共同成功标准完全相同才算完成任务
baseline_success = baseline_action == expected_action

print(f"模型动作：{baseline_action}")
print(f"正确动作：{expected_action}")
print(f"任务成功：{baseline_success}")

模型动作：restart_service
正确动作：inspect_config
任务成功：False


输出直接显示基线版本是否完成任务。失败不是因为 API 没有响应，而是尾部保留器丢掉了当前规则，只留下日志中的过时建议；最后查看这次真实决定的运行信息。

## 5.5 查看基线请求信息
本节展示基线最终决定使用的 Provider、模型、Token、停止原因和延迟，使任务失败与 API 运行状态可以分开观察。

In [15]:
# choice 记录模型正常停止生成的原因
baseline_choice = baseline_response.choices[0]
# usage 记录这次基线决定实际消耗的 Token
baseline_usage = baseline_response.usage

print(f"Provider：{config['NANO_BACKEND']}")
print(f"Model：{model_name}")
print(f"停止原因：{baseline_choice.finish_reason}")
print(f"输入 Token：{baseline_usage.prompt_tokens}")
print(f"输出 Token：{baseline_usage.completion_tokens}")
print(f"总 Token：{baseline_usage.total_tokens}")
print(f"等待时间：{baseline_latency_ms} ms")

Provider：openai
Model：LongCat-2.0
停止原因：stop
输入 Token：32
输出 Token：38
总 Token：70
等待时间：2094 ms


输出表明真实 API 正常返回并产生了完整用量数据，但任务结果仍然错误。这正是 Context Window 的核心故障：模型没有报错，外层程序却在请求前静默删掉了决定正确行为所需的信息。

# 6. 定义改进组件
## 6.1 定义大结果外置器
长工具结果不应该原样占满模型窗口。当前生产系统常把完整结果放到窗口外的对象存储或状态存储中，只在上下文保留关键事实和引用。本节用一个字典模拟窗口外存储，实现这个最小机制。

In [16]:
external_store = {}

def externalize_tool_result(content, summary):
    # 完整工具结果保存在模型上下文之外
    reference = "tool_result_1"
    external_store[reference] = content
    # 窗口内只返回关键事实和完整结果的引用
    return f"{summary}\n完整结果引用：{reference}"

print("大结果外置器已就绪")

大结果外置器已就绪


输出说明大结果外置器已经定义，但还没有移动日志。它不会删除原始结果，而是让模型只读取短摘要，需要时仍可通过引用找到完整内容；下一步定义如何在固定预算内选择上下文。

## 6.2 定义优先级选择器
仅从尾部截断无法区分内容价值。更可靠的做法是先为每项内容标出优先级，让关键规则、工具描述、当前任务和事实摘要先获得预算；某个大块放不下时跳过它，而不是停止处理全部更早内容。

In [17]:
def select_context(items, max_chars):
    # 先按优先级从高到低分配有限的窗口预算
    ranked_items = sorted(items, key=lambda item: item["priority"], reverse=True)
    selected_items = []
    used_chars = 0

    for item in ranked_items:
        item_chars = len(item["content"])
        if used_chars + item_chars <= max_chars:
            selected_items.append(item)
            used_chars += item_chars

    # 选择完成后恢复原始顺序，避免打乱模型阅读流程
    selected_items.sort(key=lambda item: items.index(item))
    return selected_items

print("优先级选择器已就绪")

优先级选择器已就绪


输出说明优先级选择器已经定义，但还没有处理上下文。它先决定哪些内容值得进入窗口，再恢复这些内容原来的先后顺序；下一章将把长日志外置，并在与基线完全相同的预算下运行修复版本。

# 7. 展示修复结果
## 7.1 外置完整日志
完整日志必须可追溯，但不必全部进入模型窗口。本节把第 3 章得到的原始日志交给外置器，窗口内只留下与当前决定有关的关键事实和引用。

In [18]:
# 原始日志完整保存到模型上下文之外
tool_summary = externalize_tool_result(
    tool_result,
    "关键事实：DB_HOST is missing",
)
# 模型窗口只接收短摘要和原始结果引用
print(f"窗口外完整日志：{len(external_store['tool_result_1'])} 字符")
print(f"窗口内摘要：{tool_summary}")

窗口外完整日志：1085 字符
窗口内摘要：关键事实：DB_HOST is missing
完整结果引用：tool_result_1


输出显示 1085 字符的完整日志仍被保存，而模型只需读取一条短摘要和引用。下一步把摘要与系统规则、工具描述、用户任务组成带优先级的上下文。

## 7.2 标出内容优先级
系统规则、当前任务和关键事实直接决定行为，因此优先级最高；工具描述负责解释可用能力，优先级次之。本节只标出这些关系，不改变内容原来的顺序。

In [19]:
# priority 数字越大，越早获得窗口预算
fixed_items = [
    {"name": "系统规则", "content": messages[0]["content"], "priority": 4},
    {"name": "工具描述", "content": json.dumps(tools, ensure_ascii=False), "priority": 3},
    {"name": "用户任务", "content": messages[1]["content"], "priority": 4},
    {"name": "工具结果摘要", "content": tool_summary, "priority": 4},
]

# 逐项展示优先级，方便观察选择器的输入
for item in fixed_items:
    print(f"{item['name']}：priority={item['priority']}，{len(item['content'])} 字符")

系统规则：priority=4，56 字符
工具描述：priority=3，139 字符
用户任务：priority=4，55 字符
工具结果摘要：priority=4，44 字符


输出显示三类决定行为的内容拥有最高优先级，工具描述紧随其后。下一步使用与基线完全相同的 350 字符预算选择并组装上下文。

## 7.3 构建改进上下文
本节运行优先级选择器，再把选中的内容按原始顺序拼接。窗口预算仍然是第 5 章定义的 `350` 字符，唯一变化是外层程序如何管理上下文。

In [20]:
# 使用与基线版本完全相同的窗口预算
fixed_selected_items = select_context(fixed_items, context_budget)
fixed_context = ""
fixed_used_chars = 0

# 按原始顺序组装模型最终读取的上下文
for item in fixed_selected_items:
    fixed_context += f"[{item['name']}]\n{item['content']}\n"
    fixed_used_chars += len(item["content"])
    print(f"保留内容：{item['name']}")

print(f"窗口预算：{context_budget} 字符")
print(f"已用预算：{fixed_used_chars} 字符")

保留内容：系统规则
保留内容：工具描述
保留内容：用户任务
保留内容：工具结果摘要
窗口预算：350 字符
已用预算：294 字符


输出显示系统规则、工具描述、用户任务和关键事实摘要全部进入了同一个 350 字符预算。完整日志仍在窗口外可供引用；下一步把这份上下文发送给同一个真实模型。

## 7.4 获取改进决定
本节沿用基线版本的回答格式、模型和温度，只把输入替换为改进组件构建的上下文。这样最终行为差异只能来自 Context Window 管理方式。

In [21]:
# 回答格式与基线请求完全相同
fixed_messages = [
    {"role": "system", "content": baseline_messages[0]["content"]},
    {"role": "user", "content": fixed_context},
]

# 使用同一个真实模型，并记录修复版本的等待时间
fixed_started = perf_counter()
fixed_response = client.chat.completions.create(
    model=model_name,
    messages=fixed_messages,
    temperature=0,
)
fixed_latency_ms = round((perf_counter() - fixed_started) * 1000)
print("改进决定已返回")

改进决定已返回


输出说明真实大模型已经基于改进上下文给出决定。下一步使用与基线完全相同的成功标准判断结果。

## 7.5 判断改进结果
本节读取真实模型选择的动作，并与第 2 章固定的 `inspect_config` 比较。只有两者完全相同，改进版本才算完成任务。

In [22]:
# 最终动作来自修复后的真实模型响应
fixed_action = fixed_response.choices[0].message.content.strip()
# 继续使用基线版本完全相同的正确答案
fixed_success = fixed_action == expected_action

print(f"模型动作：{fixed_action}")
print(f"正确动作：{expected_action}")
print(f"任务成功：{fixed_success}")

模型动作：inspect_config
正确动作：inspect_config
任务成功：True


输出直接显示改进版本是否完成任务。模型现在能够同时看到当前规则和关键事实，因此不会再把日志末尾的过时建议当成有效指令；最后查看本次真实决定的运行信息。

## 7.6 查看改进请求信息
本节展示改进最终决定使用的 Provider、模型、Token、停止原因和延迟。第 8 章会把这些数据与基线放在同一组字段中比较。

In [23]:
# choice 记录修复版本正常停止生成的原因
fixed_choice = fixed_response.choices[0]
# usage 记录修复版本实际消耗的 Token
fixed_usage = fixed_response.usage

print(f"Provider：{config['NANO_BACKEND']}")
print(f"Model：{model_name}")
print(f"停止原因：{fixed_choice.finish_reason}")
print(f"输入 Token：{fixed_usage.prompt_tokens}")
print(f"输出 Token：{fixed_usage.completion_tokens}")
print(f"总 Token：{fixed_usage.total_tokens}")
print(f"等待时间：{fixed_latency_ms} ms")

Provider：openai
Model：LongCat-2.0
停止原因：stop
输入 Token：139
输出 Token：67
总 Token：206
等待时间：2722 ms


输出记录了改进版本的真实 API 用量与延迟。至此，两条路径已经在相同任务、原始日志、窗口预算、模型和温度下完成运行；下一章只汇总现有结果。

# 8. 汇总消融对照
## 8.1 对比两种上下文管理方式
第一次日志工具请求是两条路径共享的共同成本。本节只比较产生最终动作的两次真实请求，并使用相同字段展示窗口预算、保留来源、动作、成功状态、Token 和延迟。

In [24]:
# 分别收集两条路径实际保留的上下文来源
baseline_sources = []
for item in baseline_items:
    baseline_sources.append(item["name"])

fixed_sources = []
for item in fixed_selected_items:
    fixed_sources.append(item["name"])

# 两行数据只使用已经完成的真实运行，不重新请求模型
comparison_rows = [
    {
        "版本": "尾部保留基线",
        "保留来源": "、".join(baseline_sources),
        "模型动作": baseline_action,
        "任务成功": baseline_success,
        "输入Token": baseline_usage.prompt_tokens,
        "延迟ms": baseline_latency_ms,
    },
    {
        "版本": "外置加优先级",
        "保留来源": "、".join(fixed_sources),
        "模型动作": fixed_action,
        "任务成功": fixed_success,
        "输入Token": fixed_usage.prompt_tokens,
        "延迟ms": fixed_latency_ms,
    },
]

print(f"共同窗口预算：{context_budget} 字符")
for row in comparison_rows:
    print(row)

共同窗口预算：350 字符
{'版本': '尾部保留基线', '保留来源': '工具结果尾部', '模型动作': 'restart_service', '任务成功': False, '输入Token': 32, '延迟ms': 2094}
{'版本': '外置加优先级', '保留来源': '系统规则、工具描述、用户任务、工具结果摘要', '模型动作': 'inspect_config', '任务成功': True, '输入Token': 139, '延迟ms': 2722}


对照结果直接印证了核心机制：尾部保留基线只看到过时建议，因此选择 `restart_service` 并失败；改进版本在同一预算内保留规则、工具描述、任务和事实摘要，因此选择 `inspect_config` 并成功。变化来自外层 Context Window 管理，而不是更换模型或任务。

## 8.2 拓展

### nano 版省略了什么

nano 版用字符近似 Token，并按固定优先级组装少量片段，没有供应商 Tokenizer、图像与附件成本、KV-cache、前缀稳定性、引用追踪、动态预算和上下文污染检测。生产 Context Builder 还需测量长上下文位置效应，并为被删内容留下可恢复摘要或来源指针。

### 延伸阅读

1. 2024, [Anthropic, Contextual Retrieval](https://www.anthropic.com/news/contextual-retrieval)：为检索块补充局部上下文以降低召回错误。
2. 2024, [OpenAI, Prompt Caching](https://openai.com/index/api-prompt-caching/)：稳定前缀、缓存命中与上下文成本之间的关系。
3. 2024, [LongMemEval](https://arxiv.org/abs/2410.10813)：长对话中的信息提取、时序推理和选择性遗忘。